In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import numpy as np
import copy
import seaborn as sns
import torchvision.transforms.functional as F
from model import build_resnet as build_resnet_bn
from dataset import MultiModalDataset

# ==========================================
# ==========================================

def train_transform_complex(image, tabular_data, label, raw_v_delta, label_min, label_max):
    augmented_samples = []
    # 标签归一化
    label_normalized = (label - label_min) / (label_max - label_min)
    label_tensor = torch.tensor([label_normalized], dtype=torch.float32)
    image = image.convert("L")
    post_crop_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
    for _ in range(5):
        i, j, h, w = transforms.RandomCrop.get_params(image, output_size=(336, 336))
        cropped_image = F.crop(image, i, j, h, w)
        img_rotated = F.rotate(cropped_image, 180)
        img_mirrored = F.hflip(cropped_image)
        augmented_samples.append((post_crop_transform(cropped_image), tabular_data, label_tensor, raw_v_delta))
        augmented_samples.append((post_crop_transform(img_rotated), tabular_data, label_tensor, raw_v_delta))
        augmented_samples.append((post_crop_transform(img_mirrored), tabular_data, label_tensor, raw_v_delta))
    return augmented_samples

def val_test_transform_simple(image, tabular_data, label, raw_v_delta, label_min, label_max):
    label_normalized = (label - label_min) / (label_max - label_min)
    label_tensor = torch.tensor([label_normalized], dtype=torch.float32)
    image = image.convert("L").resize((336, 336))
    image_tensor = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean=[0.5], std=[0.5])])(image)
    return [(image_tensor, tabular_data, label_tensor, raw_v_delta)]

def custom_collate_fn(batch):
    flattened_batch = [item for sublist in batch for item in sublist]
    images, tabular_data, labels, raw_v_deltas = zip(*flattened_batch)
    return torch.stack(images), torch.stack(tabular_data), torch.stack(labels), torch.stack(raw_v_deltas)

# ==========================================
# ==========================================

def train_val_test_data_process(image_dir, csv_file, v_delta_col_name='V_delta', batch_size=4, random_seed=42):
    df = pd.read_csv(csv_file)
    v_idx = df.columns.get_loc(v_delta_col_name) if v_delta_col_name in df.columns else None
    
    # 执行 Log 变换 (对应论文 Eq.1)
    df.iloc[:, -1] = np.log(df.iloc[:, -1])
    label_min, label_max = df.iloc[:, -1].min(), df.iloc[:, -1].max()
    
    df_shuffled = df.sample(frac=1, random_state=random_seed).reset_index(drop=True)
    train_size, val_size = int(0.7 * len(df_shuffled)), int(0.15 * len(df_shuffled))
    train_df = df_shuffled.iloc[:train_size].reset_index(drop=True)
    val_df = df_shuffled.iloc[train_size:train_size + val_size].reset_index(drop=True)
    test_df = df_shuffled.iloc[train_size + val_size:].reset_index(drop=True)
    
    train_ds = MultiModalDataset(
        image_dir=image_dir, 
        dataframe_slice=train_df, 
        v_delta_column_index=v_idx, 
        transform=train_transform_complex, 
        label_min=label_min, 
        label_max=label_max
    )
    
    scaler = train_ds.scaler # 提取训练集 fit 好的标准化器
    
    val_ds = MultiModalDataset(
        image_dir=image_dir, 
        dataframe_slice=val_df, 
        v_delta_column_index=v_idx, 
        transform=val_test_transform_simple, 
        label_min=label_min, 
        label_max=label_max, 
        scaler=scaler
    )
    
    test_ds = MultiModalDataset(
        image_dir=image_dir, 
        dataframe_slice=test_df, 
        v_delta_column_index=v_idx, 
        transform=val_test_transform_simple, 
        label_min=label_min, 
        label_max=label_max, 
        scaler=scaler
    )
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)
    
    return train_loader, val_loader, test_loader, label_min, label_max, v_idx

# ==========================================
# 3. 迁移学习与维度对齐逻辑
# ==========================================

def freeze_shallow_layers(model):
    for name, param in model.named_parameters():
        if any(x in name for x in ['conv1', 'bn1', 'layer1', 'layer2']):
            param.requires_grad = False
        else: param.requires_grad = True

def unfreeze_all_layers(model):
    for param in model.parameters(): param.requires_grad = True

def align_tabular_dim(batch_tabs, target_dim=29):
    curr_dim = batch_tabs.shape[1]
    if curr_dim < target_dim:
        padding = torch.zeros((batch_tabs.shape[0], target_dim - curr_dim), device=batch_tabs.device)
        return torch.cat([batch_tabs, padding], dim=1)
    elif curr_dim > target_dim:
        return batch_tabs[:, :target_dim]
    return batch_tabs

def train_model_process(model, train_loader, val_loader, test_loader,
                        label_min, label_max, v_idx, L_max, alpha, v_opt, lambda_reg,
                        num_epoch, avg_true):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.SmoothL1Loss()
    best_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    
    print("\n🚀 Stage 1: 冻结浅层阶段 (Epoch 1-50)")
    freeze_shallow_layers(model)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=5)

    for epoch in range(num_epoch):
        if epoch == 50:
            print("\n🚀 Stage 2: 全局微调阶段 (Epoch 51-100)")
            unfreeze_all_layers(model)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=5)

        model.train()
        train_l, train_samples = 0.0, 0
        all_train_out, all_train_lbl = [], []
        
        for imgs, tabs, lbls, v_deltas in train_loader:
            imgs, tabs, lbls, v_deltas = imgs.to(device).float(), tabs.to(device).float(), lbls.to(device).float().squeeze(-1), v_deltas.to(device).float()
            
            tabs = align_tabular_dim(tabs, target_dim=29)
            
            optimizer.zero_grad()
            out_norm, phys_norm, _, _, _ = model(imgs, tabs, v_deltas, L_max=L_max, alpha=alpha, v_delta_opt=v_opt, label_min=label_min, label_max=label_max)
            loss = criterion(out_norm, lbls.unsqueeze(-1)) + lambda_reg * criterion(out_norm, phys_norm)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_l += loss.item() * lbls.size(0)
            out_r = torch.exp(out_norm.squeeze(-1) * (label_max - label_min) + label_min)
            lbl_r = torch.exp(lbls * (label_max - label_min) + label_min)
            all_train_out.append(out_r.cpu().detach().numpy()); all_train_lbl.append(lbl_r.cpu().numpy())
            train_samples += lbls.size(0)
        model.eval()
        val_l, val_samples = 0.0, 0
        all_val_out, all_val_lbl = [], []
        with torch.no_grad():
            for imgs, tabs, lbls, v_deltas in val_loader:
                imgs, tabs, lbls, v_deltas = imgs.to(device).float(), tabs.to(device).float(), lbls.to(device).float().squeeze(-1), v_deltas.to(device).float()
                tabs = align_tabular_dim(tabs, target_dim=29)
                out_norm, phys_norm, _, _, _ = model(imgs, tabs, v_deltas, L_max=L_max, alpha=alpha, v_delta_opt=v_opt, label_min=label_min, label_max=label_max)
                val_l += (criterion(out_norm, lbls.unsqueeze(-1)) + lambda_reg * criterion(out_norm, phys_norm)).item() * lbls.size(0)
                out_r = torch.exp(out_norm.squeeze(-1) * (label_max - label_min) + label_min)
                lbl_r = torch.exp(lbls * (label_max - label_min) + label_min)
                all_val_out.append(out_r.cpu().detach().numpy()); all_val_lbl.append(lbl_r.cpu().numpy())
                val_samples += lbls.size(0)
        
        curr_val_loss = val_l / val_samples
        val_r2 = r2_score(np.concatenate(all_val_lbl), np.concatenate(all_val_out))
        print(f"Epoch {epoch+1} | Val Loss: {curr_val_loss:.4f} | Val R2: {val_r2:.4f}")
        
        scheduler.step(curr_val_loss)
        if curr_val_loss < best_loss:
            best_loss = curr_val_loss
            best_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_wts)
    return {'val_r2': val_r2}

# ==========================================
# 4. 主函数
# ==========================================

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    image_dir, csv_file = './GHdata', './data.csv'
    batch_size, num_epoch = 12, 100 
    L_max_phys, alpha_phys, v_opt, lambda_reg = 19, 0.001, 0.05, 0.1
    v_delta_name = 'δcontent(after)'

    train_l, val_l, test_l, l_min, l_max, v_idx = train_val_test_data_process(image_dir, csv_file, v_delta_name, batch_size)

    model = build_resnet_bn(tabular_input_dim=29).to(device)
    
    res = train_model_process(model, train_l, val_l, test_l, l_min, l_max, v_idx, L_max_phys, alpha_phys, v_opt, lambda_reg, num_epoch, 0)
    print(f"\n🎉 运行完成！验证集最终 R2: {res['val_r2']:.4f}")

if __name__ == '__main__':
    main()


✅ 系统就绪。执行关键字参数修复版复现流程...

🚀 Stage 1: 冻结浅层阶段 (Epoch 1-50)
Epoch 1 | Val Loss: 5.9869 | Val R2: 0.4812
